In [ ]:
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass



In [ ]:
cd ..

In [ ]:
from __future__ import annotations

import re
import json
import hashlib
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Optional, Dict, Iterable, Tuple

from pypdf import PdfReader


# =========================
# CONFIG (comme tu as demandé)
# =========================
PDF_PATHS: List[Path] = [
    Path("./data/in/temps_du_travail/Cadrage national DIR_2009.pdf"),
    Path("./data/in/temps_du_travail/instruction_ministerielle_du_6_janvier_2011.pdf"),
    Path("./data/in/temps_du_travail/Reglement_interieur_ARTT_AC_01012013-10.pdf"),
]

OUT_JSONL = Path(os.getenv("MATTE_AMELIORATION_CLEAN_JSONL", "./data/out/chunked/matte_temps_travail_3pdf_clean.jsonl"))

# Provenance (table rag_chunks_3)
SOURCE_ORG = "MATTE"  # ta "seule source" ministérielle ici
LANG = "fr"

# Chunking
MAX_CHARS = 1800
OVERLAP = 220
MIN_CHUNK_CHARS = 200  # évite des micro-chunks
MAX_SECTION_DEPTH = 4  # section_path garde les 4 derniers niveaux


# =========================
# Helpers
# =========================
def sha1_hex(s: str) -> str:
    return hashlib.sha1((s or "").encode("utf-8")).hexdigest()

def normalize_text(s: str) -> str:
    s = (s or "").replace("\r\n", "\n").replace("\r", "\n")
    # Nettoyage lignes
    lines = []
    for ln in s.split("\n"):
        ln = re.sub(r"[ \t]+", " ", ln).strip()
        lines.append(ln)
    s = "\n".join(lines)
    # Réduit les trous
    s = re.sub(r"\n{3,}", "\n\n", s).strip()
    return s

def compute_thematique(path: Path) -> str:
    """
    conforme à ta logique : dossier sous data/in ou data/out, sans le filename
    ex: ./data/in/temps_du_travail/X.pdf -> "temps_du_travail"
    """
    parts = [p for p in path.as_posix().split("/") if p not in ("", ".")]
    lower = [p.lower() for p in parts]
    if "data" in lower:
        i = lower.index("data")
        tail = parts[i + 1 :]
    else:
        tail = parts
    if tail and tail[0].lower() in ("in", "out"):
        tail = tail[1:]
    if tail and "." in tail[-1]:
        tail = tail[:-1]
    return "/".join(tail).strip("/")

def hard_window(text: str, max_chars: int, overlap: int) -> List[str]:
    if not text:
        return []
    if max_chars <= 0:
        return [text]
    step = max(1, max_chars - overlap)
    out = []
    i = 0
    n = len(text)
    while i < n:
        out.append(text[i : i + max_chars])
        i += step
    return out

def split_paragraph_aware(text: str, max_chars: int, overlap: int) -> List[str]:
    """
    Empile des paragraphes jusqu'à max_chars, puis fallback fenêtre dure si un paragraphe est énorme.
    """
    text = normalize_text(text)
    if not text:
        return []
    paras = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]
    chunks: List[str] = []
    buf = ""
    for p in paras:
        extra = 2 if buf else 0
        if len(buf) + extra + len(p) <= max_chars:
            buf = f"{buf}\n\n{p}" if buf else p
        else:
            if buf:
                chunks.append(buf)
            if len(p) > max_chars:
                chunks.extend(hard_window(p, max_chars, overlap))
                buf = ""
            else:
                buf = p
    if buf:
        chunks.append(buf)

    # Overlap “propre” entre chunks (approx) : on repasse en fenêtre dure si besoin
    final: List[str] = []
    for c in chunks:
        if len(c) <= max_chars:
            final.append(c)
        else:
            final.extend(hard_window(c, max_chars, overlap))
    return final

def looks_like_table(block: str) -> bool:
    """
    Heuristique simple :
    - beaucoup d'espaces alignés (colonnes)
    - ou des pipes
    - ou des lignes répétées très “courtes” type tableau
    """
    if "|" in block and block.count("|") >= 2:
        return True
    lines = [ln for ln in block.split("\n") if ln.strip()]
    if len(lines) >= 3:
        # alignements / colonnes
        multi_space = sum(1 for ln in lines if re.search(r"\S\s{3,}\S", ln))
        if multi_space >= 2:
            return True
        # “lignes tableau” assez courtes
        shortish = sum(1 for ln in lines if 5 <= len(ln) <= 80)
        if shortish / max(1, len(lines)) > 0.75:
            return True
    return False


# =========================
# PDF extraction (page by page)
# =========================
def read_pdf_pages(path: Path) -> List[str]:
    reader = PdfReader(str(path))
    pages_text: List[str] = []
    for i, page in enumerate(reader.pages, start=1):
        t = (page.extract_text() or "").strip()
        t = normalize_text(t)
        if t:
            pages_text.append(f"[PAGE {i}]\n{t}")
        else:
            pages_text.append(f"[PAGE {i}]\n")  # garde la trace
    return pages_text

def read_pdf_fulltext(path: Path) -> str:
    return normalize_text("\n\n".join(read_pdf_pages(path)))


# =========================
# Sectioning (documents “règlement / instruction / circulaire”)
# =========================
HEAD_PAT = re.compile(
    r"^(?:"
    r"Préambule\b|"
    r"Table des matières\b|"
    r"(?:Annexe|ANNEXE)\s*\d+[\.\-–]?\s*.*|"
    r"\d+\s*[-–]\s*.+|"
    r"\d+\.\d+\s*[-–]\s*.+|"
    r"[IVXLC]+\s*[-–\.]\s*.+"
    r")$",
    re.IGNORECASE,
)

# repère les lignes type “2 - Modalités …” et “2.1 - …”
NUM_HEAD = re.compile(r"^(\d+(?:\.\d+)*)\s*[-–]\s*(.+)$")

ANNEXE_HEAD = re.compile(r"^(Annexe|ANNEXE)\s*(\d+)\s*[\.\-–]?\s*(.*)$")

def build_sections(text: str) -> List[Tuple[str, str]]:
    """
    Retourne [(section_path, section_text)].
    - section_path : hiérarchie (best-effort)
    - section_text : contenu “sous” le titre
    """
    lines = text.split("\n")

    section_stack: List[str] = []
    current_title: Optional[str] = None
    current_buf: List[str] = []

    out: List[Tuple[str, str]] = []

    def flush():
        nonlocal current_title, current_buf
        if current_title is None:
            return
        body = normalize_text("\n".join(current_buf).strip())
        # évite sections vides
        if body:
            path = " > ".join(section_stack[-MAX_SECTION_DEPTH:])
            out.append((path, body))
        current_buf = []

    for raw in lines:
        ln = raw.strip()
        if not ln:
            if current_title is not None:
                current_buf.append("")
            continue

        # ignore page markers as headings
        if ln.startswith("[PAGE "):
            if current_title is not None:
                current_buf.append(ln)
            continue

        if HEAD_PAT.match(ln):
            # nouveau heading => flush précédent
            flush()

            # Normalise titre
            title = ln
            m_num = NUM_HEAD.match(ln)
            if m_num:
                num, label = m_num.group(1), m_num.group(2).strip()
                title = f"{num} - {label}"

            m_ann = ANNEXE_HEAD.match(ln)
            if m_ann:
                an_num = m_ann.group(2)
                an_title = (m_ann.group(3) or "").strip()
                title = f"Annexe {an_num}" + (f" - {an_title}" if an_title else "")

            # logique stack : si numéroté “2.1” => niveau = nb de points + 1
            if m_num:
                num = m_num.group(1)
                level = num.count(".") + 1
                section_stack = section_stack[: level - 1]
                section_stack.append(title)
            elif m_ann:
                # annexes : on “reset” un niveau “Annexes”
                # (ça évite qu’elles se retrouvent sous “11.2 …” etc.)
                base = "Annexes"
                # enlève éventuel ancien “Annexes”
                section_stack = [s for s in section_stack if s.lower() != base.lower()]
                section_stack.append(base)
                section_stack.append(title)
            else:
                # Préambule / Table des matières / roman etc. => push simple
                # garde seulement les derniers niveaux
                section_stack.append(title)
                section_stack = section_stack[-MAX_SECTION_DEPTH:]

            current_title = title
            current_buf = []
        else:
            if current_title is None:
                # texte avant le premier titre : on le colle dans une section “Document”
                current_title = "Document"
                section_stack = ["Document"]
                current_buf = [ln]
            else:
                current_buf.append(ln)

    flush()
    return out


# =========================
# RAG rows (rag_chunks_3)
# =========================
@dataclass
class RagRow:
    hash_id: str
    qa_id: str
    parent_qa_id: Optional[str]
    source_name: str
    section_path: str
    role: str
    chunk_index: int
    text: str
    lang: str
    thematique: str
    source: str

def make_hash_id(source_name: str, section_path: str, role: str, chunk_index: int, text: str) -> str:
    key = f"{source_name}|{section_path}|{role}|{chunk_index}|{text}"
    return sha1_hex(key)

def section_qa_id(source_name: str, section_path: str) -> str:
    # stable: un id par section
    return sha1_hex(f"{source_name}|{section_path}".lower())

def emit_rows_for_pdf(pdf_path: Path) -> Iterable[RagRow]:
    full = read_pdf_fulltext(pdf_path)

    thematique = compute_thematique(pdf_path)
    source_name = pdf_path.name

    sections = build_sections(full)
    if not sections:
        sections = [("Document", full)]

    for sec_path, sec_text in sections:
        qa_id = section_qa_id(source_name, sec_path)
        parent_qa = None  # documents non Q/A : pas de parentage strict ici

        # 1) un “header chunk” (utile pour retrieval)
        header_text = f"TITRE/SECTION: {sec_path}\n\n{sec_text[: max(500, min(len(sec_text), 1200)) ]}"
        header_text = normalize_text(header_text)
        if len(header_text) >= MIN_CHUNK_CHARS:
            role = "SECTION_HEADER"
            idx = 0
            hid = make_hash_id(source_name, sec_path, role, idx, header_text)
            yield RagRow(
                hash_id=hid,
                qa_id=qa_id,
                parent_qa_id=parent_qa,
                source_name=source_name,
                section_path=sec_path,
                role=role,
                chunk_index=idx,
                text=header_text,
                lang=LANG,
                thematique=thematique,
                source=SOURCE_ORG,
            )

        # 2) chunks principaux (paragraph-aware)
        chunks = split_paragraph_aware(sec_text, MAX_CHARS, OVERLAP)
        for i, ch in enumerate(chunks, start=1):
            ch = normalize_text(ch)
            if len(ch) < MIN_CHUNK_CHARS:
                continue
            role = "CHUNK"
            hid = make_hash_id(source_name, sec_path, role, i, ch)
            yield RagRow(
                hash_id=hid,
                qa_id=qa_id,
                parent_qa_id=parent_qa,
                source_name=source_name,
                section_path=sec_path,
                role=role,
                chunk_index=i,
                text=ch,
                lang=LANG,
                thematique=thematique,
                source=SOURCE_ORG,
            )

            # 3) si ça ressemble à un tableau, on duplique en role TABLE (meilleur recall)
            if looks_like_table(ch):
                role_t = "TABLE"
                hid_t = make_hash_id(source_name, sec_path, role_t, i, ch)
                yield RagRow(
                    hash_id=hid_t,
                    qa_id=qa_id,
                    parent_qa_id=parent_qa,
                    source_name=source_name,
                    section_path=sec_path,
                    role=role_t,
                    chunk_index=i,
                    text=ch,
                    lang=LANG,
                    thematique=thematique,
                    source=SOURCE_ORG,
                )


# =========================
# MAIN : JSONL stream (anti-crash)
# =========================
def main():
    OUT_JSONL.parent.mkdir(parents=True, exist_ok=True)

    total = 0
    with OUT_JSONL.open("w", encoding="utf-8") as f:
        for pdf in PDF_PATHS:
            if not pdf.exists():
                print(f"[WARN] PDF introuvable: {pdf}")
                continue

            print(f"→ Processing: {pdf}")
            for row in emit_rows_for_pdf(pdf):
                f.write(json.dumps(asdict(row), ensure_ascii=False) + "\n")
                total += 1

    print(f"\n✓ JSONL écrit: {OUT_JSONL}")
    print(f"✓ Total rows: {total}")
    print("Done.")


if __name__ == "__main__":
    main()


In [ ]:
from sentence_transformers import SentenceTransformer
import torch, numpy as np
import os, json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import fastparquet  # noqa
engine = "fastparquet"

MODEL_NAME = os.getenv("EMBEDDING_MODEL", "BAAI/bge-m3")

BATCH_SIZE = 64                  # adjust to RAM/VRAM
NORMALIZE  = True                # cosine-ready vectors
EMBED_COL  = os.getenv("EMBEDDING_COLUMN", "embedding_m3")



device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)
print(f"Loaded {MODEL_NAME} on {device}")

def format_passage(text: str) -> str:
    """Model-aware passage formatting."""
    if MODEL_NAME.startswith("intfloat/multilingual-e5"):
        return f"passage: {text or ''}"
    # bge-m3 / gte-multilingual-base: no instruction needed
    return text or ""

# ==========================================================
# Fixed & improved embeddings writer (robust, model-aware)
# ==========================================================


# ---------- Config / defaults ----------
IN_JSONL    = os.getenv("MATTE_AMELIORATION_IN_JSONL", "./data/out/chunked/matte_temps_travail_3pdf_clean.jsonl")
TAG         = (MODEL_NAME.replace("/", "_").replace("-", "_")).lower()
OUT_PARQUET = os.getenv("MATTE_AMELIORATION_OUT_PARQUET", f"./data/out/matte_temps_du_travail_amelioration_chunks_{TAG}.parquet")
OUT_NPY     = os.getenv("MATTE_AMELIORATION_OUT_NPY", f"./data/out/matte_temps_du_travail_amelioration_chunks_{TAG}.npy")
OUT_JSONL   = os.getenv("MATTE_AMELIORATION_OUT_JSONL_WITH_EMB", f"./data/out/matte_temps_du_travail_amelioration_chunks_{TAG}_with_emb.jsonl")
# ---------- Helpers ----------
def sha1_u(s: str) -> str:
    return hashlib.sha1((s or "").encode("utf-8")).hexdigest()

def make_hash_id(row) -> str:
    key = f"{row.get('source_name','')}|{row.get('qa_id','')}|{row.get('role','')}|{row.get('chunk_index',0)}|{(row.get('text','')[:256])}"
    return sha1_u(key)

# ---------- Load chunks ----------
rows = []
with open(IN_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))
df = pd.DataFrame(rows)
assert not df.empty, f"Le fichier {IN_JSONL} est vide."

# Stable ids
df["hash_id"] = df.apply(make_hash_id, axis=1)

# ---------- Build corpus (model-aware) ----------
corpus = [format_passage(t) for t in df["text"].astype(str).tolist()]

# ---------- Encode in batches ----------
vecs_all = []
for i in range(0, len(corpus), BATCH_SIZE):
    batch = corpus[i:i+BATCH_SIZE]
    vecs = model.encode(
        batch,
        batch_size=len(batch),           # respect small leftover batch
        convert_to_numpy=True,
        normalize_embeddings=NORMALIZE,
        show_progress_bar=True,
    )
    vecs_all.append(vecs)
emb = np.vstack(vecs_all).astype(np.float32)  # (N, d)

# attach to df
df[EMBED_COL] = [v.tolist() for v in emb]
df['source']="SERVICE PUBLIC"
# ---------- Save Parquet (embeddings as JSON strings) ----------
Path(OUT_PARQUET).parent.mkdir(parents=True, exist_ok=True)
df_parquet = df.copy()
df_parquet[EMBED_COL] = df_parquet[EMBED_COL].apply(json.dumps)  # store as JSON string




   
if engine is None:
    print("⚠️ Neither 'fastparquet' nor 'pyarrow' found. Skipping Parquet save.")
else:
    df_parquet.to_parquet(OUT_PARQUET, index=False, engine=engine)
    print(f"Parquet saved to: {OUT_PARQUET} (engine={engine}, embeddings as JSON strings)")

# ---------- Save NPY matrix ----------
np.save(OUT_NPY, emb)
print("NPY matrix saved to:", OUT_NPY, emb.shape)

# ---------- Save JSONL with embeddings inline ----------
Path(OUT_JSONL).parent.mkdir(parents=True, exist_ok=True)
with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for rec in df.to_dict(orient="records"):
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print("JSONL+emb saved to:", OUT_JSONL)

# ---------- Peek ----------
print(df[["source_name","role","section_path","chunk_index","hash_id"]].head(10).to_string(index=False))
